# NPR + ResNet18 From Scratch - Kaggle Folder Tiny-GenImage

Notebook reads Tiny-GenImage folder data from Kaggle Input and trains a random-initialized ResNet18 on NPR residual maps.

Flow:

```text
image -> NPR residual -> ResNet18(weights=None) -> real/fake
```

Config numbers are matched with the comparison experiments: full data, batch size 32, max 10 epochs, patience 3, learning rate 2e-4, weight decay 1e-4.


## 0. Install dependencies


In [ ]:
%pip install -q torchvision scikit-learn pandas tqdm


## 1. Import and find project root


In [ ]:
import sys
from pathlib import Path


def find_code_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(kaggle_input.glob("*"))
        candidates.extend(kaggle_input.glob("*/*"))
        candidates.extend(kaggle_input.glob("*/*/*"))
        for init_file in kaggle_input.rglob("__init__.py"):
            if init_file.parent.name == "data_loader":
                return init_file.parent.parent

    for candidate in candidates:
        if (candidate / "data_loader" / "__init__.py").exists():
            return candidate

    print("/kaggle/input children:")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.glob("*")):
            print(" -", path)
    raise FileNotFoundError("Cannot find HoangHa_Code/data_loader.")


CODE_ROOT = find_code_root()
PROJECT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else CODE_ROOT
sys.path.insert(0, str(CODE_ROOT))
print("CODE_ROOT =", CODE_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)


import gc
import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18

from data_loader import (
    TinyGenImageKaggleConfig,
    TinyGenImageKaggleDataset,
    build_image_transform,
    build_kaggle_tiny_index,
    build_kaggle_tiny_splits,
    collate_unified_batch,
    find_tiny_genimage_root,
    summarize_index,
)
import baselines.npr_resnet18.npr_resnet18 as npr_train_utils
from baselines.npr_resnet18.npr_resnet18 import (
    build_kaggle_loaders,
    make_output_root,
    seed_everything,
    train_eval_experiment,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = torch.cuda.is_available()
USE_AMP = torch.cuda.is_available()
print("DEVICE =", DEVICE)


## 2. Config


In [ ]:
DATASET_ROOT = None
RUN_ALL_CASES = False
SELECTED_EXPERIMENT = "combined"
EXPERIMENT_CONFIGS = [
    {"name": "combined", "eval_case": "combined"},
    {"name": "in_domain_biggan", "eval_case": "in_domain", "generator": "BigGAN"},
    {"name": "cross_generator_glide", "eval_case": "cross_generator", "heldout_generator": "GLIDE"},
    {"name": "cross_generator_wukong", "eval_case": "cross_generator", "heldout_generator": "Wukong"},
    {"name": "train_one_generator_biggan", "eval_case": "train_one_generator", "base_generator": "BigGAN"},
]

BALANCE_REAL = True
RANDOM_SEED = 42
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None
VAL_FRACTION = 0.2

BATCH_SIZE = 32
NUM_WORKERS = 0
MAX_EPOCHS = 10
PATIENCE = 3
MIN_DELTA = 1e-3
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4

SAVE_PREDICTIONS = True
OUTPUT_ROOT, RUN_ID = make_output_root(PROJECT_ROOT, "npr_resnet18_kaggle_folder")
print("RUN_ID =", RUN_ID)


## 3. NPR + ResNet18 model definition

Cell n?y l? ph?n ki?n tr?c ch?nh c?a baseline: ?nh RGB ???c bi?n ??i th?nh residual b?ng NPR, sau ?? ??a v?o ResNet18 kh?i t?o ng?u nhi?n (`weights=None`). Stem m?c ??nh ???c ??i t? `7x7 stride=2 + maxpool` sang `3x3 stride=1 + Identity` ?? gi? chi ti?t local pixel artifacts.


In [ ]:
class NPRLayer(nn.Module):
    def __init__(self, factor=0.5, scale=2.0 / 3.0):
        super().__init__()
        self.factor = factor
        self.scale = scale

    def forward(self, x):
        _, _, height, width = x.shape
        if height % 2 == 1:
            x = x[:, :, :-1, :]
        if width % 2 == 1:
            x = x[:, :, :, :-1]

        down = F.interpolate(
            x,
            scale_factor=self.factor,
            mode="nearest",
            recompute_scale_factor=True,
        )
        up = F.interpolate(down, size=x.shape[-2:], mode="nearest")
        return (x - up) * self.scale


class NPRResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.npr = NPRLayer()
        self.backbone = resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.backbone.maxpool = nn.Identity()
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        residual = self.npr(x)
        return self.backbone(residual)


def make_notebook_model(device):
    model = NPRResNet18(num_classes=2).to(device)
    print("Model: NPRLayer -> ResNet18(weights=None), train from scratch")
    return model


dummy_batch = torch.zeros(2, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    dummy_logits = make_notebook_model(DEVICE)(dummy_batch)
print("Dummy logits shape:", tuple(dummy_logits.shape))
del dummy_batch, dummy_logits
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Make train_eval_experiment use the model defined in this notebook.
# This works with both old and new npr_resnet18.py utility files.
npr_train_utils.make_model = make_notebook_model
print("Training utility make_model patched to notebook NPRResNet18")


## 4. Build splits, loaders, and run helpers


In [ ]:
def build_splits_for_experiment(exp, detected_root):
    split_config = TinyGenImageKaggleConfig(
        dataset_root=str(detected_root),
        eval_case=exp["eval_case"],
        generator=exp.get("generator"),
        heldout_generator=exp.get("heldout_generator"),
        base_generator=exp.get("base_generator"),
        balance_real=BALANCE_REAL,
        seed=RANDOM_SEED,
        max_train_samples=MAX_TRAIN_SAMPLES,
        max_eval_samples=MAX_TEST_SAMPLES,
    )
    splits = build_kaggle_tiny_splits(split_config)
    print(f"\n=== {exp['name']} ===")
    print(splits["notes"])
    print("train real/fake:", splits["train_real_count"], splits["train_fake_count"])
    print("test real/fake:", splits["eval_real_count"], splits["eval_fake_count"])
    return splits


def run_experiment(exp, detected_root):
    seed_everything(RANDOM_SEED)
    splits = build_splits_for_experiment(exp, detected_root)
    train_loader, val_loader, test_loader, splits = build_kaggle_loaders(
        splits=splits,
        dataset_class=TinyGenImageKaggleDataset,
        collate_fn=collate_unified_batch,
        transform_train=build_image_transform(image_size=224, train=True),
        transform_eval=build_image_transform(image_size=224, train=False),
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        val_fraction=VAL_FRACTION,
        seed=RANDOM_SEED,
    )
    print("inner train/val:", len(splits["train_inner_df"]), len(splits["val_inner_df"]))

    run_dir = OUTPUT_ROOT / exp["name"] / RUN_ID
    (run_dir / "metrics").mkdir(parents=True, exist_ok=True)
    splits["train_inner_df"].to_csv(run_dir / "metrics" / "train_inner_split.csv", index=False)
    splits["val_inner_df"].to_csv(run_dir / "metrics" / "val_inner_split.csv", index=False)
    splits["eval_df"].to_csv(run_dir / "metrics" / "test_split.csv", index=False)

    config_summary = {
        "dataset_root": splits["dataset_root"],
        "balance_real": BALANCE_REAL,
        "seed": RANDOM_SEED,
        "max_train_samples": MAX_TRAIN_SAMPLES,
        "max_test_samples": MAX_TEST_SAMPLES,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "min_delta": MIN_DELTA,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "train_rows": splits["train_rows"],
        "test_rows": splits["eval_rows"],
    }
    return train_eval_experiment(
        exp=exp,
        run_dir=run_dir,
        loaders=(train_loader, val_loader, test_loader),
        splits=splits,
        config_summary=config_summary,
        device=DEVICE,
        use_amp=USE_AMP,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        save_predictions=SAVE_PREDICTIONS,
    )


def main():
    detected_root = find_tiny_genimage_root(DATASET_ROOT)
    print("Detected Tiny-GenImage root:", detected_root)
    index_df = build_kaggle_tiny_index(TinyGenImageKaggleConfig(dataset_root=str(detected_root)))
    structure_path = OUTPUT_ROOT / f"dataset_structure_summary_{RUN_ID}.csv"
    summarize_index(index_df).to_csv(structure_path, index=False)
    print("Dataset structure summary saved:", structure_path)
    print("Generators:", sorted(index_df["generator"].unique().tolist()))

    experiments = EXPERIMENT_CONFIGS if RUN_ALL_CASES else [
        exp for exp in EXPERIMENT_CONFIGS if exp["name"] == SELECTED_EXPERIMENT
    ]
    if not experiments:
        raise ValueError(f"Unknown SELECTED_EXPERIMENT={SELECTED_EXPERIMENT!r}")

    all_metrics = [run_experiment(exp, detected_root) for exp in experiments]
    summary_df = pd.DataFrame(all_metrics)
    summary_path = OUTPUT_ROOT / f"summary_metrics_{RUN_ID}.csv"
    summary_df.to_csv(summary_path, index=False)
    print("Summary saved:", summary_path)
    display(summary_df[["experiment_name", "eval_case", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "average_precision", "best_epoch"]])


## 5. Run


In [ ]:
main()
